# BICOL-WEAVE-AI — MobileNetV2 Training

Phase 1 — AI / Model, Step 2: dataset preparation and MobileNetV2 transfer-learning pipeline.

This notebook is self-contained for Google Colab. It audits the inherited Google Drive dataset, excludes exact duplicates according to the documented policy, creates a stratified split, trains MobileNetV2, evaluates the best checkpoint, and writes all generated outputs to Google Drive.

## Section 1 — Environment Setup

In [ ]:
import os
import sys
import hashlib
import json
import time
import random
from pathlib import Path
from typing import Dict, List, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'Selected device: {device}')
if torch.cuda.is_available():
    print(f'CUDA GPU: {torch.cuda.get_device_name(0)}')

## Section 2 — Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DATASET_DIR = Path('/content/drive/MyDrive/Coconut_Weave_Dataset')
OUTPUT_DIR = Path('/content/drive/MyDrive/BICOL-WEAVE-AI-OUTPUTS')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['plain', 'twill', 'complex']
CLASS_TO_IDX = {class_name: index for index, class_name in enumerate(CLASS_NAMES)}
CLASS_CODES = {'plain': 'S1', 'twill': 'S2', 'complex': 'S3'}
IMAGE_SIZE = 224

missing_class_dirs = [
    class_name for class_name in CLASS_NAMES
    if not (DATASET_DIR / class_name).is_dir()
]
if missing_class_dirs:
    raise FileNotFoundError(
        f'Missing required dataset directories under {DATASET_DIR}: {missing_class_dirs}'
    )

print(f'Dataset directory: {DATASET_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Class order: {CLASS_NAMES}')

## Section 3 — Dataset Audit

The manifest keeps every supported source file. Same-class duplicate groups keep one deterministic file, while every occurrence of a hash shared across classes is excluded as a label conflict. No files are deleted from Google Drive.

In [ ]:
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

def sha256_file(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with file_path.open('rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def build_raw_manifest(dataset_dir: Path) -> pd.DataFrame:
    rows = []
    for class_name in CLASS_NAMES:
        class_dir = dataset_dir / class_name
        for file_path in sorted(class_dir.rglob('*')):
            if file_path.is_file() and file_path.suffix.lower() in SUPPORTED_EXTENSIONS:
                rows.append({
                    'filepath': str(file_path),
                    'filename': file_path.name,
                    'class_name': class_name,
                    'file_size': file_path.stat().st_size,
                    'sha256': sha256_file(file_path),
                })
    return pd.DataFrame(rows, columns=['filepath', 'filename', 'class_name', 'file_size', 'sha256'])

def classify_duplicate_manifest(manifest: pd.DataFrame) -> pd.DataFrame:
    audited = manifest.copy()
    audited['status'] = 'usable'
    audited['duplicate_group_size'] = audited.groupby('sha256')['sha256'].transform('size')
    for _, group in audited.groupby('sha256', sort=True):
        if group['class_name'].nunique() > 1:
            audited.loc[group.index, 'status'] = 'excluded_cross_class_conflict'
        elif len(group) > 1:
            keep_index = group.sort_values('filepath').index[0]
            duplicate_indexes = group.index.difference([keep_index])
            audited.loc[duplicate_indexes, 'status'] = 'excluded_same_class_duplicate'
    return audited

raw_manifest = build_raw_manifest(DATASET_DIR)
if raw_manifest.empty:
    raise ValueError('No supported images were found in the required class directories.')

print(f'Total raw images: {len(raw_manifest)}')
for class_name in CLASS_NAMES:
    print(f'{class_name.title()} count: {(raw_manifest["class_name"] == class_name).sum()}')

audited_manifest = classify_duplicate_manifest(raw_manifest)
usable_manifest = audited_manifest[audited_manifest['status'] == 'usable'].copy()
usable_manifest = usable_manifest.sort_values('filepath').reset_index(drop=True)
same_class_excluded = int((audited_manifest['status'] == 'excluded_same_class_duplicate').sum())
cross_class_excluded = int((audited_manifest['status'] == 'excluded_cross_class_conflict').sum())
print(f'Raw image count: {len(audited_manifest)}')
print(f'Same-class duplicate files excluded: {same_class_excluded}')
print(f'Cross-class conflicting files excluded: {cross_class_excluded}')
print(f'Final usable image count: {len(usable_manifest)}')
print('Usable count per class:')
print(usable_manifest['class_name'].value_counts().reindex(CLASS_NAMES, fill_value=0).to_string())

dataset_manifest_path = OUTPUT_DIR / 'dataset_manifest.csv'
audited_manifest.to_csv(dataset_manifest_path, index=False)
print(f'Saved complete manifest: {dataset_manifest_path}')

## Section 4 — Train/Test Split

Only usable images enter the split. The test set uses the deterministic test transform defined in the next section and never receives training augmentation.

In [ ]:
usable_counts = usable_manifest['class_name'].value_counts()
if (usable_counts.reindex(CLASS_NAMES, fill_value=0) < 2).any():
    raise ValueError('Each class needs at least two usable images for a stratified 80/20 split.')

train_manifest, test_manifest = train_test_split(
    usable_manifest,
    test_size=0.20,
    random_state=SEED,
    stratify=usable_manifest['class_name'],
)
train_manifest = train_manifest.copy()
test_manifest = test_manifest.copy()
train_manifest['split'] = 'train'
test_manifest['split'] = 'test'
split_manifest = pd.concat([train_manifest, test_manifest], ignore_index=True)
split_manifest['label'] = split_manifest['class_name'].map(CLASS_TO_IDX).astype(int)
split_manifest = split_manifest.sort_values(['split', 'class_name', 'filepath']).reset_index(drop=True)
train_manifest = split_manifest[split_manifest['split'] == 'train'].copy()
test_manifest = split_manifest[split_manifest['split'] == 'test'].copy()

split_summary = (
    split_manifest.groupby(['class_name', 'split']).size()
    .unstack(fill_value=0)
    .reindex(CLASS_NAMES)
    .reindex(columns=['train', 'test'], fill_value=0)
)
split_summary['total'] = split_summary['train'] + split_summary['test']
split_summary = split_summary.reset_index().rename(columns={'class_name': 'class'})
print(split_summary[['class', 'total', 'train', 'test']].to_string(index=False))

dataset_split_path = OUTPUT_DIR / 'dataset_split.csv'
split_manifest.to_csv(dataset_split_path, index=False)
print(f'Saved split manifest: {dataset_split_path}')

## Section 5 — Dataset Class

The final model input is 224 × 224 RGB. Training uses light spatial augmentation; test images use only deterministic resizing, center cropping, tensor conversion, and ImageNet normalization.

In [ ]:
from torch.utils.data import DataLoader, Dataset

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
BATCH_SIZE = 16

class WeaveDataset(Dataset):
    def __init__(self, records: pd.DataFrame, transform: transforms.Compose) -> None:
        self.records = records.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        row = self.records.iloc[index]
        with Image.open(row['filepath']) as image:
            image_rgb = image.convert('RGB')
            image_tensor = self.transform(image_rgb)
        return image_tensor, int(row['label'])

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = WeaveDataset(train_manifest, train_transform)
test_dataset = WeaveDataset(test_manifest, test_transform)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
    pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
    pin_memory=torch.cuda.is_available()
)
print(f'Train images: {len(train_dataset)}')
print(f'Test images: {len(test_dataset)}')
print(f'Batch size: {BATCH_SIZE}')

## Section 6 — Data Visualization

In [ ]:
images, labels = next(iter(train_loader))
sample_count = min(8, len(images))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
mean_tensor = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std_tensor = torch.tensor(IMAGENET_STD).view(3, 1, 1)
for index, axis in enumerate(axes):
    axis.axis('off')
    if index < sample_count:
        image = images[index].cpu() * std_tensor + mean_tensor
        image = image.clamp(0, 1).permute(1, 2, 0)
        axis.imshow(image)
        axis.set_title(CLASS_NAMES[int(labels[index])])
fig.suptitle('Training samples with training augmentation')
plt.tight_layout()
plt.show()

## Section 7 — MobileNetV2

Class order is fixed: `0 = plain`, `1 = twill`, `2 = complex`. The feature extractor is frozen for the first five epochs, then the full model is fine-tuned.

In [ ]:
def create_mobilenet_v2(pretrained: bool = True) -> nn.Module:
    weights = torchvision.models.MobileNet_V2_Weights.DEFAULT if pretrained else None
    model = torchvision.models.mobilenet_v2(weights=weights)
    input_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(input_features, len(CLASS_NAMES))
    return model

model = create_mobilenet_v2(pretrained=True).to(device)
for parameter in model.features.parameters():
    parameter.requires_grad = False

LEARNING_RATE = 1e-4
EPOCHS = 30
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda parameter: parameter.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
)
print(f'Model: MobileNetV2')
print(f'Number of output classes: {len(CLASS_NAMES)}')
print(f'Class mapping: {CLASS_TO_IDX}')

## Section 8 — Training Loop

For this presentation MVP, the holdout test split is evaluated after every epoch for monitoring and best-checkpoint selection. This is acceptable for the MVP but is not ideal research methodology. The final research version should use a separate validation set or cross-validation for model selection.

In [ ]:
def evaluate_accuracy(model: nn.Module, data_loader: DataLoader) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.inference_mode():
        for batch_images, batch_labels in data_loader:
            batch_images = batch_images.to(device, non_blocking=True)
            batch_labels = batch_labels.to(device, non_blocking=True)
            logits = model(batch_images)
            predictions = logits.argmax(dim=1)
            correct += (predictions == batch_labels).sum().item()
            total += batch_labels.size(0)
    return correct / total if total else 0.0

BEST_MODEL_PATH = OUTPUT_DIR / 'mobilenet_v2_best.pth'
best_test_accuracy = -1.0
history = {'epoch': [], 'train_loss': [], 'train_accuracy': [], 'test_accuracy': []}

for epoch in range(1, EPOCHS + 1):
    if epoch == 6:
        for parameter in model.features.parameters():
            parameter.requires_grad = True
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        print('Feature extractor unfrozen; fine-tuning all parameters.')

    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0
    for batch_images, batch_labels in train_loader:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(batch_images)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_labels.size(0)
        running_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        running_total += batch_labels.size(0)

    train_loss = running_loss / running_total
    train_accuracy = running_correct / running_total
    test_accuracy = evaluate_accuracy(model, test_loader)
    is_best = test_accuracy > best_test_accuracy
    if is_best:
        best_test_accuracy = test_accuracy
        checkpoint = {
            'model_state_dict': model.state_dict(),
            'class_names': CLASS_NAMES,
            'class_to_idx': CLASS_TO_IDX,
            'image_size': IMAGE_SIZE,
            'normalization_mean': IMAGENET_MEAN,
            'normalization_std': IMAGENET_STD,
            'epoch': epoch,
            'accuracy': float(test_accuracy),
        }
        torch.save(checkpoint, BEST_MODEL_PATH)

    history['epoch'].append(epoch)
    history['train_loss'].append(float(train_loss))
    history['train_accuracy'].append(float(train_accuracy))
    history['test_accuracy'].append(float(test_accuracy))
    print(f'Epoch {epoch:02d}/{EPOCHS}')
    print(f'Train Loss: {train_loss:.4f}')
    print(f'Train Accuracy: {train_accuracy:.4f}')
    print(f'Test Accuracy: {test_accuracy:.4f}')
    print(f'Best Model: {"Yes" if is_best else "No"}')

print(f'Best test accuracy: {best_test_accuracy:.4f}')
print(f'Saved best model: {BEST_MODEL_PATH}')

## Section 9 — Training Curves

In [ ]:
history_df = pd.DataFrame(history)

def save_curve(y_column: str, title: str, ylabel: str, filename: str) -> None:
    plt.figure(figsize=(8, 5))
    plt.plot(history_df['epoch'], history_df[y_column], marker='o')
    plt.xlabel('Epoch')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    output_path = OUTPUT_DIR / filename
    plt.savefig(output_path, dpi=160)
    plt.show()
    plt.close()
    print(f'Saved: {output_path}')

save_curve('train_loss', 'Training Loss vs Epoch', 'Training Loss', 'training_loss.png')
save_curve('train_accuracy', 'Training Accuracy vs Epoch', 'Training Accuracy', 'training_accuracy.png')
save_curve('test_accuracy', 'Test Accuracy vs Epoch', 'Test Accuracy', 'test_accuracy.png')

## Section 10 — Final Evaluation

The best checkpoint is reloaded before final evaluation. Metrics are computed on the full holdout test split, and inference timing uses batch size 1 with CUDA warm-up and synchronization when CUDA is available.

In [ ]:
best_checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
best_model = create_mobilenet_v2(pretrained=False).to(device)
best_model.load_state_dict(best_checkpoint['model_state_dict'])
best_model.eval()

def collect_predictions(model: nn.Module, data_loader: DataLoader):
    targets = []
    predictions = []
    model.eval()
    with torch.inference_mode():
        for batch_images, batch_labels in data_loader:
            logits = model(batch_images.to(device, non_blocking=True))
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
            targets.extend(batch_labels.tolist())
    return targets, predictions

test_targets, test_predictions = collect_predictions(best_model, test_loader)
final_accuracy = float(accuracy_score(test_targets, test_predictions))
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    test_targets,
    test_predictions,
    labels=list(range(len(CLASS_NAMES))),
    average='macro',
    zero_division=0,
)
classification_report_dict = classification_report(
    test_targets,
    test_predictions,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
print(classification_report(
    test_targets, test_predictions, labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES, zero_division=0
))

def synchronize_cuda() -> None:
    if device.type == 'cuda':
        torch.cuda.synchronize()

single_test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)
def mean_inference_time_ms(model: nn.Module, data_loader: DataLoader) -> float:
    model.eval()
    if device.type == 'cuda':
        warmup_images, _ = next(iter(data_loader))
        warmup_images = warmup_images[:1].to(device)
        for _ in range(5):
            with torch.inference_mode():
                model(warmup_images)
        synchronize_cuda()

    timings = []
    for batch_images, _ in data_loader:
        batch_images = batch_images.to(device, non_blocking=True)
        synchronize_cuda()
        start_time = time.perf_counter()
        with torch.inference_mode():
            model(batch_images)
        synchronize_cuda()
        timings.append((time.perf_counter() - start_time) * 1000.0)
    return float(np.mean(timings)) if timings else 0.0

inference_ms = mean_inference_time_ms(best_model, single_test_loader)
print(f'Model: MobileNetV2')
print(f'Test Images: {len(test_targets)}')
print(f'Accuracy: {final_accuracy:.6f}')
print(f'Macro Precision: {macro_precision:.6f}')
print(f'Macro Recall: {macro_recall:.6f}')
print(f'Macro F1: {macro_f1:.6f}')
print(f'Average Inference Time: {inference_ms:.3f} ms')

## Section 11 — Save Results

In [ ]:
results = {
    'model': 'MobileNetV2',
    'classes': CLASS_NAMES,
    'train_count': int(len(train_dataset)),
    'test_count': int(len(test_dataset)),
    'accuracy': final_accuracy,
    'macro_precision': float(macro_precision),
    'macro_recall': float(macro_recall),
    'macro_f1': float(macro_f1),
    'inference_ms': float(inference_ms),
}

results_path = OUTPUT_DIR / 'mobilenet_results.json'
with results_path.open('w', encoding='utf-8') as file_handle:
    json.dump(results, file_handle, indent=2)

report_path = OUTPUT_DIR / 'classification_report.csv'
report_rows = []
for report_label, report_values in classification_report_dict.items():
    if isinstance(report_values, dict):
        report_rows.append({'label': report_label, **report_values})
    else:
        report_rows.append({'label': report_label, 'accuracy': float(report_values)})
report_dataframe = pd.DataFrame(report_rows).set_index('label')
report_dataframe.to_csv(report_path)

confusion = confusion_matrix(test_targets, test_predictions, labels=list(range(len(CLASS_NAMES))))
confusion_path = OUTPUT_DIR / 'confusion_matrix.png'
plt.figure(figsize=(7, 6))
plt.imshow(confusion, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(CLASS_NAMES))
plt.xticks(tick_marks, CLASS_NAMES, rotation=45, ha='right')
plt.yticks(tick_marks, CLASS_NAMES)
threshold = confusion.max() / 2.0 if confusion.size else 0.0
for row_index in range(confusion.shape[0]):
    for column_index in range(confusion.shape[1]):
        plt.text(column_index, row_index, confusion[row_index, column_index],
                 ha='center', va='center',
                 color='white' if confusion[row_index, column_index] > threshold else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(confusion_path, dpi=160)
plt.show()
plt.close()

print(f'Saved results: {results_path}')
print(f'Saved classification report: {report_path}')
print(f'Saved confusion matrix: {confusion_path}')

## Section 12 — Single Image Prediction

This function uses the reloaded best checkpoint and computes all returned values from the image. Confidence and probabilities are softmax outputs, not hardcoded values.

In [ ]:
def predict_image(image_path: Union[str, Path]) -> Dict[str, object]:
    image_path = Path(image_path)
    with Image.open(image_path) as image:
        image_rgb = image.convert('RGB')
        input_tensor = test_transform(image_rgb).unsqueeze(0).to(device)

    best_model.eval()
    synchronize_cuda()
    start_time = time.perf_counter()
    with torch.inference_mode():
        logits = best_model(input_tensor)
        probability_tensor = torch.softmax(logits, dim=1)[0]
    synchronize_cuda()

    inference_time = (time.perf_counter() - start_time) * 1000.0
    predicted_index = int(probability_tensor.argmax().item())
    prediction = CLASS_NAMES[predicted_index]
    probabilities = {
        class_name: float(probability_tensor[index].item())
        for index, class_name in enumerate(CLASS_NAMES)
    }
    return {
        'prediction': prediction,
        'class_code': CLASS_CODES[prediction],
        'confidence': probabilities[prediction],
        'probabilities': probabilities,
        'inference_ms': float(inference_time),
    }

## Section 13 — Upload Test

Optional emergency backup demo for Colab if the website is unavailable. Upload one JPG, JPEG, or PNG image.

In [ ]:
from google.colab import files

uploaded = files.upload()
uploaded_images = [
    Path(name) for name in uploaded
    if Path(name).suffix.lower() in {'.jpg', '.jpeg', '.png'}
]
if not uploaded_images:
    raise ValueError('Please upload one JPG, JPEG, or PNG image.')

uploaded_path = uploaded_images[0]
prediction_result = predict_image(uploaded_path)
with Image.open(uploaded_path) as uploaded_image:
    plt.figure(figsize=(7, 5))
    plt.imshow(uploaded_image.convert('RGB'))
    plt.axis('off')
    plt.title(f"Prediction: {prediction_result['prediction']} ({prediction_result['class_code']})")
    plt.show()

print(f"Prediction: {prediction_result['prediction']}")
print(f"Class code: {prediction_result['class_code']}")
print(f"Confidence: {prediction_result['confidence'] * 100:.2f}%")
print('Probability distribution:')
for class_name, probability in prediction_result['probabilities'].items():
    print(f'  {class_name}: {probability * 100:.2f}%')
print(f"Inference time: {prediction_result['inference_ms']:.3f} ms")